# 🤖 Üretken Yapay Zeka — Transformer ve Büyük Dil Modelleri
## Transformer · Encoder-Decoder · Attention · LLM
### 4. Ders Uygulamalı Notebook

---
**İçindekiler:**
1. RNN vs Transformer: Gradyan Kaybolması
2. Encoder–Decoder ve Bilgi Darboğazı
3. Bahdanau (Additive) Dikkat Mekanizması
4. Scaled Dot-Product Attention (Q, K, V)
5. Multi-Head Attention
6. Pozisyonel Kodlama (Sinüzoidal, RoPE, ALiBi)
7. Feed-Forward Network & Aktivasyon Fonksiyonları
8. Layer Normalization: LayerNorm vs RMSNorm / Pre-LN vs Post-LN
9. Dikkat Maskeleme: Full vs Causal
10. Tam Transformer Bloğu (Sıfırdan Implementasyon)
11. Mini GPT: Karakter Seviyesi Dil Modeli
12. Hiperparametre Analizi & Ölçekleme Yasası

In [ ]:
# Gerekli kurulumlar
!pip install torch matplotlib numpy scipy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import math
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {device} | PyTorch: {torch.__version__}')

---
## Bölüm 1: RNN vs Transformer — Gradyan Kaybolması Problemi

RNN'lerin temel sorunu:
- Adım adım işlem → **paralel hesaplama imkânsız**
- Uzun dizilerde **gradyan kaybolması/patlaması**
- Sabit boyutlu hidden state → **bilgi darboğazı**

In [ ]:
# -----------------------------------------------------------
# 1.1 — RNN Gradyan Kaybolması: Sayısal Gösterim
# Gradyanın dizi boyunca nasıl küçüldüğünü göster
# -----------------------------------------------------------
def simulate_rnn_gradient(seq_len=50, W_hh_val=0.9):
    """
    Basit RNN'de gradyan büyüklüğünü simüle eder.
    dL/dh_t ∝ W_hh^(T-t)  → üstel küçülme!
    """
    grad_norms = []
    for t in range(seq_len):
        # |dL/dh_t| ≈ |W_hh|^(T-t)
        grad = abs(W_hh_val) ** (seq_len - t)
        grad_norms.append(grad)
    return grad_norms

seq_len = 50
grads_vanish  = simulate_rnn_gradient(seq_len, W_hh_val=0.85)
grads_stable  = simulate_rnn_gradient(seq_len, W_hh_val=1.00)
grads_explode = simulate_rnn_gradient(seq_len, W_hh_val=1.15)

# -----------------------------------------------------------
# 1.2 — Transformer: Her adım arasında DOĞRUDAN bağlantı
# Self-Attention O(1) mesafede her token çifti ile bağlantı kurar
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('RNN Gradyan Sorunu vs Transformer Çözümü', fontsize=14, fontweight='bold')

# Gradyan kaybolması
axes[0].semilogy(range(seq_len), grads_vanish,  'r-o',  ms=4, lw=2, label='|W_hh|=0.85 → Kaybolma')
axes[0].semilogy(range(seq_len), grads_stable,  'g-s',  ms=4, lw=2, label='|W_hh|=1.00 → Stabil')
axes[0].semilogy(range(seq_len), grads_explode, 'b-^',  ms=4, lw=2, label='|W_hh|=1.15 → Patlama')
axes[0].set_xlabel('Token pozisyonu (başa uzaklık)')
axes[0].set_ylabel('Gradyan büyüklüğü (log ölçek)')
axes[0].set_title('RNN Gradyan Akışı\n|dL/dh_t| ≈ |W_hh|^(T-t)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(1e-4, color='gray', ls='--', alpha=0.5, label='Pratik sıfır')

# RNN vs Transformer: Bağlantı mesafeleri
T = 8
rnn_distances    = list(range(T, 0, -1))        # Doğrusal: T, T-1, ..., 1
transf_distances = [1] * T                       # Sabit: her zaman O(1)

x = np.arange(T)
axes[1].bar(x - 0.2, rnn_distances,    0.35, label='RNN: O(T) adım',        color='salmon',    alpha=0.8)
axes[1].bar(x + 0.2, transf_distances, 0.35, label='Transformer: O(1) adım', color='steelblue', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Token {i+1}' for i in range(T)], rotation=30)
axes[1].set_ylabel('Son token\'a bağlantı mesafesi (adım)')
axes[1].set_title('Her Token\'un Son Token\'a Uzaklığı\nTransformer: Hepsine Doğrudan Erişim!')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Kilit Çıkarım:')
print(f'  RNN  : T={seq_len} dizisinde ilk token gradyanı ~ {grads_vanish[0]:.2e}')
print(f'  Trans: Her token her token\'a O(1) adımda erişir → gradyan korunur!')

---
## Bölüm 2: Encoder–Decoder ve Bilgi Darboğazı

RNN Encoder–Decoder:
$$\mathbf{h} = \text{Enc}(x_1, \ldots, x_T), \quad y_t = \text{Dec}(\mathbf{h}, y_1, \ldots, y_{t-1})$$

**Sorun:** Tüm girdi bilgisi tek bir **h** vektörüne sıkıştırılıyor!

In [ ]:
# -----------------------------------------------------------
# 2.1 — Bilgi Darboğazını Sayısal Olarak Göster
# Basit RNN Encoder ile farklı uzunluktaki dizileri kodla
# ve ne kadar bilgi kaybolduğunu ölç
# -----------------------------------------------------------
class SimpleRNNEncoder(nn.Module):
    def __init__(self, input_size=10, hidden_size=16):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.hidden_size = hidden_size

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        outputs, h = self.rnn(x)  # outputs: tüm adımlar, h: son hidden
        return outputs, h.squeeze(0)

encoder = SimpleRNNEncoder(input_size=8, hidden_size=32)

# Farklı uzunluktaki "aynı anlam" dizilerini kodla
seq_lens   = [5, 10, 20, 40, 80]
hidden_dim = 32

# Sabit referans: ilk 5 token
ref_input    = torch.randn(1, 5, 8)
_, ref_state = encoder(ref_input)

similarities = []
for slen in seq_lens:
    # Aynı 5 token + rastgele ek tokenlar
    if slen > 5:
        extra      = torch.randn(1, slen - 5, 8)
        full_input = torch.cat([ref_input, extra], dim=1)
    else:
        full_input = ref_input
    _, full_state = encoder(full_input)
    sim = F.cosine_similarity(ref_state, full_state).item()
    similarities.append(abs(sim))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Encoder–Decoder: Bilgi Darboğazı Sorunu', fontsize=14, fontweight='bold')

axes[0].plot(seq_lens, similarities, 'r-o', ms=8, lw=2)
axes[0].set_xlabel('Dizi Uzunluğu')
axes[0].set_ylabel('İlk 5 token\'la Kosinüs Benzerliği')
axes[0].set_title('RNN Hidden State Bilgi Kaybı\nDizi uzadıkça başlangıç bilgisi kaybolur!')
axes[0].axhline(1.0, color='green', ls='--', label='Mükemmel hafıza')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.1)

# Mimari karşılaştırma: bottleneck vs attention
ax = axes[1]
ax.axis('off')

# RNN Encoder-Decoder
for i, label in enumerate(['x₁','x₂','x₃','...','xT']):
    ax.add_patch(mpatches.FancyBboxPatch([0.02+i*0.07, 0.65], 0.055, 0.08,
        boxstyle='round,pad=0.01', fc='steelblue', ec='white', alpha=0.8))
    ax.text(0.047+i*0.07, 0.69, label, ha='center', color='white', fontsize=8, fontweight='bold')

ax.add_patch(mpatches.FancyBboxPatch([0.18, 0.45], 0.08, 0.1,
    boxstyle='round,pad=0.01', fc='red', ec='white', alpha=0.9))
ax.text(0.22, 0.50, 'h\n(darboğaz!)', ha='center', va='center', color='white', fontsize=9, fontweight='bold')

for i, label in enumerate(['y₁','y₂','y₃']):
    ax.add_patch(mpatches.FancyBboxPatch([0.32+i*0.07, 0.65], 0.055, 0.08,
        boxstyle='round,pad=0.01', fc='green', ec='white', alpha=0.8))
    ax.text(0.347+i*0.07, 0.69, label, ha='center', color='white', fontsize=8, fontweight='bold')

ax.annotate('', xy=(0.18, 0.50), xytext=(0.10, 0.65),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(0.32, 0.69), xytext=(0.26, 0.50),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.text(0.22, 0.38, '❌ RNN Enc-Dec: Tek vektör darboğazı', ha='center', color='red', fontsize=10)

# Attention mekanizması
ax.text(0.70, 0.80, '✅ Attention: Tüm encoder\nçıktılarına doğrudan erişim',
        ha='center', va='center', fontsize=10, color='darkgreen',
        bbox=dict(boxstyle='round', fc='lightgreen', alpha=0.5))
ax.text(0.70, 0.55, 'ct = Σ αt,i · hi\n\nHer decoder adımında\ngirdi dizisine yeniden bak!',
        ha='center', va='center', fontsize=11, fontfamily='monospace',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

ax.set_xlim(0, 1); ax.set_ylim(0.2, 1)
ax.set_title('Mimari Karşılaştırma', fontsize=11)

plt.tight_layout()
plt.show()

---
## Bölüm 3: Bahdanau (Additive) Dikkat Mekanizması

**3 Adımda Dikkat:**

1. **Skor:** $e_{t,i} = v^\top \tanh(W_s s_{t-1} + W_h h_i)$
2. **Normalize:** $\alpha_{t,i} = \dfrac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}$
3. **Bağlam:** $c_t = \sum_i \alpha_{t,i} h_i$

In [ ]:
# -----------------------------------------------------------
# 3.1 — Bahdanau Dikkat: Sıfırdan Implementasyon
# -----------------------------------------------------------
class BahdanauAttention(nn.Module):
    """
    Bahdanau (2014) Additive Attention
    e_{t,i} = v^T * tanh(W_s * s_{t-1} + W_h * h_i)
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.W_s = nn.Linear(hidden_size, hidden_size, bias=False)  # decoder state
        self.W_h = nn.Linear(hidden_size, hidden_size, bias=False)  # encoder states
        self.v   = nn.Linear(hidden_size, 1, bias=False)            # scoring vector

    def forward(self, s_prev, encoder_outputs):
        """
        s_prev:          (batch, hidden)   — decoder'ın önceki durumu
        encoder_outputs: (batch, T, hidden) — tüm encoder çıktıları
        """
        # s_prev'i her encoder adımıyla eşleştir
        s_exp = s_prev.unsqueeze(1).expand_as(encoder_outputs)  # (B, T, H)

        # Adım 1: Alignment skorları
        energy = self.v(torch.tanh(
            self.W_s(s_exp) + self.W_h(encoder_outputs)
        )).squeeze(-1)  # (B, T)

        # Adım 2: Softmax → ağırlıklar
        alpha = F.softmax(energy, dim=-1)  # (B, T), toplamı 1

        # Adım 3: Bağlam vektörü
        context = torch.bmm(alpha.unsqueeze(1), encoder_outputs).squeeze(1)  # (B, H)

        return context, alpha

# -----------------------------------------------------------
# 3.2 — Türkçe→Almanca çeviri senaryosu (simülasyon)
# "Transformerlar harikadır" → dikkat ağırlıklarını görselleştir
# -----------------------------------------------------------
H = 16
attn = BahdanauAttention(H)

src_tokens  = ['Trans', 'are', 'great', '.']          # Kaynak (İngilizce)
tgt_tokens  = ['Trans', 'sind', 'großartig', '.']     # Hedef (Almanca)
T_src, T_tgt = len(src_tokens), len(tgt_tokens)

# Rastgele encoder çıktıları
encoder_out = torch.randn(1, T_src, H)

# Her decoder adımı için dikkat ağırlıklarını hesapla
attention_matrix = np.zeros((T_tgt, T_src))
s = torch.zeros(1, H)  # başlangıç decoder durumu

for t in range(T_tgt):
    ctx, alpha = attn(s, encoder_out)
    attention_matrix[t] = alpha.squeeze().detach().numpy()
    # Decoder durumunu güncelle (basit)
    s = torch.tanh(ctx + s)

# -----------------------------------------------------------
# 3.3 — Dikkat Isı Haritası
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bahdanau Dikkat Mekanizması', fontsize=14, fontweight='bold')

im = axes[0].imshow(attention_matrix, cmap='Blues', aspect='auto', vmin=0)
axes[0].set_xticks(range(T_src));  axes[0].set_xticklabels(src_tokens, fontsize=12)
axes[0].set_yticks(range(T_tgt));  axes[0].set_yticklabels(tgt_tokens, fontsize=12)
axes[0].set_xlabel('Kaynak (İngilizce)', fontsize=11)
axes[0].set_ylabel('Hedef (Almanca)',    fontsize=11)
axes[0].set_title('Dikkat Ağırlıkları α_{t,i}\nRenk = Önem derecesi')
for i in range(T_tgt):
    for j in range(T_src):
        axes[0].text(j, i, f'{attention_matrix[i,j]:.2f}',
                     ha='center', va='center', fontsize=9,
                     color='white' if attention_matrix[i,j] > 0.5 else 'black')
plt.colorbar(im, ax=axes[0], label='α (dikkat ağırlığı)')

# Bağlam vektörü hesabı görselleştirmesi
ax = axes[1]
ax.axis('off')
ax.set_title('Dikkat Mekanizması Adımları', fontsize=11)

steps = [
    ('1. Skor Hesaplama', 'e_{t,i} = vᵀ tanh(W_s·s_{t-1} + W_h·h_i)', '#3498DB'),
    ('2. Normalize (Softmax)', 'α_{t,i} = exp(e_{t,i}) / Σ exp(e_{t,j})', '#27AE60'),
    ('3. Bağlam Vektörü', 'c_t = Σ α_{t,i} · h_i', '#E74C3C'),
    ('Sezgisel Anlam', '"Bu çıktıyı üretirken\nhangi girdiye bakmalıyım?"', '#8E44AD'),
]
for i, (title, formula, color) in enumerate(steps):
    y = 0.85 - i * 0.22
    ax.add_patch(mpatches.FancyBboxPatch([0.03, y-0.08], 0.93, 0.17,
        boxstyle='round,pad=0.01', fc=color, ec='white', alpha=0.15))
    ax.text(0.50, y+0.05, title,   ha='center', fontsize=11, fontweight='bold', color=color)
    ax.text(0.50, y-0.02, formula, ha='center', fontsize=10, fontfamily='monospace', color='black')

ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.tight_layout()
plt.show()

---
## Bölüm 4: Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- **Q** (Query): "Ben ne arıyorum?"
- **K** (Key): "Bende ne var?"
- **V** (Value): "Taşıdığım bilgi"
- **√d_k**: Büyük boyutlarda nokta çarpımı patlamasını önler

In [ ]:
# -----------------------------------------------------------
# 4.1 — Scaled Dot-Product Attention: Adım Adım
# -----------------------------------------------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

    Q: (batch, heads, seq, d_k)
    K: (batch, heads, seq, d_k)
    V: (batch, heads, seq, d_v)
    mask: (seq, seq) — kausal maskesi için
    """
    d_k = Q.shape[-1]

    # Adım 1: Ham skorlar
    scores = torch.matmul(Q, K.transpose(-2, -1))          # (..., T, T)

    # Adım 2: Ölçekle (√d_k ile böl)
    scores_scaled = scores / math.sqrt(d_k)

    # Adım 3: Maske uygula (opsiyonel)
    if mask is not None:
        scores_scaled = scores_scaled.masked_fill(mask == 0, float('-inf'))

    # Adım 4: Softmax → ağırlıklar
    attn_weights = F.softmax(scores_scaled, dim=-1)         # (..., T, T)

    # Adım 5: Value ağırlıklı toplam
    output = torch.matmul(attn_weights, V)                  # (..., T, d_v)

    return output, attn_weights

# -----------------------------------------------------------
# 4.2 — √d_k Ölçeklemesinin Önemi
# -----------------------------------------------------------
print('√d_k Ölçeklemesi Neden Gerekli?')
print('=' * 50)

dk_values = [1, 4, 16, 64, 256, 1024]
unscaled_softmax_entropy = []
scaled_softmax_entropy   = []

for dk in dk_values:
    q = torch.randn(100, dk)
    k = torch.randn(100, dk)
    dots = torch.matmul(q, k.T) / dk   # dk ile normalize (ölçekleme etkisini doğru gösterir)

    # Ölçeksiz softmax
    sw_unscaled = F.softmax(dots, dim=-1)
    H_unscaled  = -(sw_unscaled * (sw_unscaled + 1e-9).log()).sum(-1).mean().item()

    # Ölçekli softmax
    sw_scaled = F.softmax(dots / math.sqrt(dk), dim=-1)
    H_scaled  = -(sw_scaled * (sw_scaled + 1e-9).log()).sum(-1).mean().item()

    unscaled_softmax_entropy.append(H_unscaled)
    scaled_softmax_entropy.append(H_scaled)
    print(f'd_k={dk:5d}: Entropy (ölçeksiz)={H_unscaled:.3f}, (ölçekli)={H_scaled:.3f}')

# -----------------------------------------------------------
# 4.3 — Görselleştirme
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Scaled Dot-Product Attention', fontsize=14, fontweight='bold')

axes[0].semilogx(dk_values, unscaled_softmax_entropy, 'r-o', ms=7, lw=2, label='Ölçeksiz (QK^T)')
axes[0].semilogx(dk_values, scaled_softmax_entropy,   'g-s', ms=7, lw=2, label='Ölçekli (QK^T/√dk)')
axes[0].set_xlabel('d_k (key boyutu)')
axes[0].set_ylabel('Softmax Entropy')
axes[0].set_title('Ölçeklemenin Önemi\nDüşük entropy = tek yere odaklanma = öğrenme güçlüğü')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Dikkat haritası örneği
T, dk = 6, 8
words = ['The', 'cat', 'sat', 'on', 'the', 'mat']
Q_ex = torch.randn(1, 1, T, dk)
K_ex = torch.randn(1, 1, T, dk)
V_ex = torch.randn(1, 1, T, dk)

_, attn_ex = scaled_dot_product_attention(Q_ex, K_ex, V_ex)
attn_map   = attn_ex.squeeze().detach().numpy()

im2 = axes[1].imshow(attn_map, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(T)); axes[1].set_xticklabels(words, fontsize=11)
axes[1].set_yticks(range(T)); axes[1].set_yticklabels(words, fontsize=11)
axes[1].set_title('Attention Haritası — Örnek\n"The cat sat on the mat"')
axes[1].set_xlabel('Key (K)'); axes[1].set_ylabel('Query (Q)')
for i in range(T):
    for j in range(T):
        axes[1].text(j, i, f'{attn_map[i,j]:.2f}', ha='center', va='center', fontsize=8,
                     color='black' if 0.2 < attn_map[i,j] < 0.8 else 'white')
plt.colorbar(im2, ax=axes[1], label='Dikkat ağırlığı')

plt.tight_layout()
plt.show()

# Boyut analizi
print('\nBoyut Analizi:')
B, T, d_model, d_k = 2, 6, 16, 8
X   = torch.randn(B, T, d_model)
W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)
Q   = X @ W_Q; K = X @ W_K; V = X @ W_V
print(f'X  : {tuple(X.shape)}   (batch={B}, T={T}, d_model={d_model})')
print(f'W_Q: {tuple(W_Q.shape)} → Q: {tuple(Q.shape)}')
print(f'QK^T: ({B}, {T}, {T})  ← her token çifti için skor')
print(f'Z  : ({B}, {T}, {d_k}) ← her token için zenginleştirilmiş temsil')

---
## Bölüm 5: Multi-Head Attention (MHA)

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$
$$\text{MHA} = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

**Fayda:** Her baş farklı bir ilişki türünü öğrenir: pozisyon, sözdizimi, anlam...

In [ ]:
# -----------------------------------------------------------
# 5.1 — Multi-Head Attention: Sıfırdan Implementasyon
# -----------------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model, n_heads\'e bölünebilmeli'
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads  # Her başın boyutu

        # Her baş için ayrı W^Q, W^K, W^V yerine
        # tek büyük matris (verimlilik için)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """(B, T, d_model) → (B, n_heads, T, d_k)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.n_heads, self.d_k)
        return x.transpose(1, 2)  # (B, n_heads, T, d_k)

    def forward(self, Q, K, V, mask=None):
        B, T, _ = Q.shape

        # Projeksiyon + baş ayrımı
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # Her baş için paralel attention
        out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # Başları birleştir
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

        # Çıktı projeksiyonu
        return self.W_o(out), attn_weights

# -----------------------------------------------------------
# 5.2 — Her Başın Farklı Şey Öğrendiğini Göster
# -----------------------------------------------------------
d_model, n_heads, T = 32, 4, 8
mha   = MultiHeadAttention(d_model, n_heads, dropout=0.0)

# Test girdisi
x_test = torch.randn(1, T, d_model)
out, attn_maps = mha(x_test, x_test, x_test)  # Self-Attention

print(f'Girdi boyutu    : {tuple(x_test.shape)}')
print(f'Çıktı boyutu    : {tuple(out.shape)}')
print(f'Attention harit. : {tuple(attn_maps.shape)} → (B, n_heads, T, T)')

# Her başın dikkat haritasını görselleştir
fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))
fig.suptitle(f'Multi-Head Attention: {n_heads} Başın Dikkat Haritaları\n'
             f'Her baş farklı ilişkilere odaklanır!', fontsize=12, fontweight='bold')

head_labels = ['Baş 1\n(Pozisyon?)', 'Baş 2\n(Sözdizimi?)',
               'Baş 3\n(Anlam?)',    'Baş 4\n(Mesafe?)']
cmaps       = ['Blues', 'Reds', 'Greens', 'Purples']

for h in range(n_heads):
    head_map = attn_maps[0, h].detach().numpy()
    axes[h].imshow(head_map, cmap=cmaps[h], aspect='auto', vmin=0)
    axes[h].set_title(head_labels[h], fontsize=10)
    axes[h].set_xlabel('Key pozisyonu')
    if h == 0: axes[h].set_ylabel('Query pozisyonu')

plt.tight_layout()
plt.show()

# Parametre analizi
total_params = sum(p.numel() for p in mha.parameters())
print(f'\nMHA Parametre Analizi (d_model={d_model}, n_heads={n_heads}):')
print(f'  W_q, W_k, W_v, W_o: her biri {d_model}×{d_model} = {d_model*d_model}')
print(f'  Toplam: {total_params:,} parametre')

---
## Bölüm 6: Pozisyonel Kodlama

Self-attention **konum bilgisi içermez** — tüm kelimeleri aynı anda işler.
Konum bilgisi dışarıdan eklenmeli!

**Sinüzoidal PE:** $PE_{(pos,2i)} = \sin\!\left(\dfrac{pos}{10000^{2i/d}}\right)$

In [ ]:
# -----------------------------------------------------------
# 6.1 — Sinüzoidal Pozisyonel Kodlama
# -----------------------------------------------------------
def sinusoidal_pe(max_len, d_model):
    """
    PE_{pos, 2i}   = sin(pos / 10000^{2i/d})
    PE_{pos, 2i+1} = cos(pos / 10000^{2i/d})
    """
    PE   = torch.zeros(max_len, d_model)
    pos  = torch.arange(max_len).unsqueeze(1).float()      # (T, 1)
    dims = torch.arange(0, d_model, 2).float()             # çift indisler
    div  = 10000 ** (dims / d_model)                       # payda terimi

    PE[:, 0::2] = torch.sin(pos / div)   # çift boyutlar
    PE[:, 1::2] = torch.cos(pos / div)   # tek boyutlar
    return PE

# -----------------------------------------------------------
# 6.2 — RoPE (Rotary Position Embedding)
# Token çiftleri arasındaki göreceli mesafeyi döndürme ile kodlar
# -----------------------------------------------------------
def rope_embedding(x, theta=10000.0):
    """
    Basitleştirilmiş RoPE: 2D rotasyon uygulaması
    q^T_m k_n ∝ f(m-n)  — göreceli pozisyon!
    """
    B, T, d = x.shape
    assert d % 2 == 0

    freqs = 1.0 / (theta ** (torch.arange(0, d, 2).float() / d))
    pos   = torch.arange(T).float()
    angles = torch.outer(pos, freqs)  # (T, d/2)

    cos_a = torch.cos(angles).unsqueeze(0)  # (1, T, d/2)
    sin_a = torch.sin(angles).unsqueeze(0)

    # x'i çift-tek çiftlere ayır
    x_even = x[:, :, 0::2]
    x_odd  = x[:, :, 1::2]

    # 2D rotasyon: [cos θ, -sin θ; sin θ, cos θ] * [x_e; x_o]
    x_rot_even = x_even * cos_a - x_odd * sin_a
    x_rot_odd  = x_even * sin_a + x_odd * cos_a

    x_rope = torch.stack([x_rot_even, x_rot_odd], dim=-1).reshape(B, T, d)
    return x_rope

# -----------------------------------------------------------
# 6.3 — ALiBi: Attention skorlarına lineer ceza
# e_{ij} = q_i^T k_j - m * |i-j|
# -----------------------------------------------------------
def alibi_bias(T, n_heads):
    """Her baş için farklı lineer ceza eğimi."""
    # Baş özel eğimler (2^{-8/n_heads})
    slopes = torch.tensor([2 ** (-8 * (i+1) / n_heads) for i in range(n_heads)])
    pos    = torch.arange(T)
    dist   = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs().float()  # (T, T)
    bias   = -slopes.view(n_heads, 1, 1) * dist.unsqueeze(0)       # (H, T, T)
    return bias

# -----------------------------------------------------------
# 6.4 — Görselleştirme
# -----------------------------------------------------------
max_len, d_model = 50, 64
PE_sin = sinusoidal_pe(max_len, d_model)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Pozisyonel Kodlama Yöntemleri', fontsize=14, fontweight='bold')

# Sinüzoidal PE matrisi
im1 = axes[0,0].imshow(PE_sin.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0,0].set_xlabel('Boyut (d)'); axes[0,0].set_ylabel('Pozisyon (pos)')
axes[0,0].set_title('Sinüzoidal PE Matrisi\nRenkler = PE değerleri')
plt.colorbar(im1, ax=axes[0,0])

# Belirli boyutların dalga formları
positions = np.arange(max_len)
for i, dim_pair in enumerate([(0,1), (4,5), (16,17), (30,31)]):
    axes[0,1].plot(positions, PE_sin[:,dim_pair[0]].numpy(),
                   label=f'sin dim={dim_pair[0]}', lw=2)
axes[0,1].set_xlabel('Pozisyon'); axes[0,1].set_ylabel('PE Değeri')
axes[0,1].set_title('Farklı Boyutlardaki Sin Dalgaları\nDüşük boyut = yüksek frekans')
axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)

# Pozisyon benzerliği (dot-product)
pe_sim = PE_sin @ PE_sin.T  # (T, T)
im3 = axes[0,2].imshow(pe_sim.numpy(), cmap='Blues', aspect='auto')
axes[0,2].set_title('PE Benzerlik Matrisi (PE·PE^T)\nYakın pozisyonlar benzer!')
axes[0,2].set_xlabel('Pozisyon'); axes[0,2].set_ylabel('Pozisyon')
plt.colorbar(im3, ax=axes[0,2])

# RoPE
x_test_rope = torch.randn(1, max_len, d_model)
x_rope      = rope_embedding(x_test_rope)
rope_sim    = (x_rope[0] @ x_rope[0].T).detach().numpy()
im4 = axes[1,0].imshow(rope_sim, cmap='RdBu', aspect='auto')
axes[1,0].set_title('RoPE — Göreceli Pozisyon Benzerliği\nDiyagonal: kendi kendine yüksek skor')
axes[1,0].set_xlabel('Pozisyon'); axes[1,0].set_ylabel('Pozisyon')
plt.colorbar(im4, ax=axes[1,0])

# ALiBi bias
T_alibi, h_alibi = 20, 4
bias = alibi_bias(T_alibi, h_alibi)
im5 = axes[1,1].imshow(bias[0].numpy(), cmap='RdBu_r', aspect='auto')
axes[1,1].set_title('ALiBi Ceza (Baş 1)\ne_{ij} = q^Tk - m·|i-j|')
axes[1,1].set_xlabel('Key pozisyon'); axes[1,1].set_ylabel('Query pozisyon')
plt.colorbar(im5, ax=axes[1,1])

# PE Karşılaştırma tablosu
axes[1,2].axis('off')
table_data = [
    ['Yöntem',        'Öğrenilir', 'Uzun bağlam', 'Göreceli', 'Kullanım'],
    ['Sinüzoidal',    'Hayır',     'Orta',         'Hayır',    'Vaswani\'17'],
    ['Öğrenilen Mut.','Evet',      'Sınırlı',      'Hayır',    'BERT, GPT'],
    ['RoPE',          'Evet',      'İyi',          'Evet',     'LLaMA, Mistral'],
    ['ALiBi',         'Hayır',     'Çok iyi',      'Evet',     'BLOOM, MPT'],
]
tbl = axes[1,2].table(cellText=table_data[1:], colLabels=table_data[0],
                       cellLoc='center', loc='center', bbox=[0,0.1,1,0.85])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for j in range(5):
    tbl[0,j].set_facecolor('#2C3E50')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
row_colors = ['#EBF5FB','#FDFEFE']
for i in range(1,5):
    for j in range(5):
        tbl[i,j].set_facecolor(row_colors[i%2])
axes[1,2].set_title('PE Yöntemleri Karşılaştırması', fontweight='bold')
axes[1,2].text(0.5, 0.02, '💡 Modern LLM\'lerde (LLaMA, Mistral) RoPE standarttır!',
               ha='center', fontsize=10, color='darkgreen', style='italic',
               transform=axes[1,2].transAxes)

plt.tight_layout()
plt.show()

---
## Bölüm 7: Feed-Forward Network & Aktivasyon Fonksiyonları

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

Her token'a **aynı** FFN uygulanır (konum bağımsız). $d_{ff} \approx 4 \cdot d_{\text{model}}$

In [ ]:
# -----------------------------------------------------------
# 7.1 — Aktivasyon Fonksiyonları
# -----------------------------------------------------------
def relu(x):  return torch.maximum(x, torch.zeros_like(x))

def gelu(x):
    """GELU(x) = x * Φ(x)  (Φ: standart normal CDF)"""
    return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2)))

def swish(x, beta=1.0):
    """Swish(x) = x * sigmoid(beta * x)"""
    return x * torch.sigmoid(beta * x)

def swiglu(x, W, V):
    """SwiGLU(x) = Swish(xW) ⊙ xV  — LLaMA-2'de kullanılır"""
    return swish(x @ W) * (x @ V)

x = torch.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('FFN Aktivasyon Fonksiyonları', fontsize=14, fontweight='bold')

# Fonksiyonlar
axes[0].plot(x, relu(x),  'r-',  lw=2.5, label='ReLU (Orijinal Transformer)')
axes[0].plot(x, gelu(x),  'b-',  lw=2.5, label='GELU (BERT, GPT-2)')
axes[0].plot(x, swish(x), 'g-',  lw=2.5, label='Swish/SiLU (SwiGLU\'ın temeli)')
axes[0].axhline(0, color='k', ls=':', alpha=0.5)
axes[0].axvline(0, color='k', ls=':', alpha=0.5)
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].set_title('Aktivasyon Fonksiyonları\nGELU = ReLU\'nun yumuşak versiyonu')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-1, 4)

# Türevler (gradyan akışı için önemli)
x_grad = x.requires_grad_(True)
grads  = {}
for name, fn in [('ReLU', relu), ('GELU', gelu), ('Swish', swish)]:
    y = fn(x_grad)
    y.sum().backward()
    grads[name] = x_grad.grad.clone().detach()
    x_grad.grad.zero_()

for name, color in [('ReLU','r'), ('GELU','b'), ('Swish','g')]:
    axes[1].plot(x.detach(), grads[name], f'{color}-', lw=2, label=f'd/dx {name}')
axes[1].axhline(0, color='k', ls=':', alpha=0.5)
axes[1].axvline(0, color='k', ls=':', alpha=0.5)
axes[1].set_xlabel('x'); axes[1].set_ylabel('df/dx')
axes[1].set_title('Gradyan Analizi\nReLU: x<0 için sıfır gradyan = "ölü nöron"')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.2, 1.2)

# FFN Genişleme
d_models = [128, 256, 512, 768, 1024, 4096]
d_ffs    = [d * 4 for d in d_models]
params_ffn = [2 * d * dff for d, dff in zip(d_models, d_ffs)]

ax_twin = axes[2].twinx()
axes[2].bar(range(len(d_models)), d_ffs, color='steelblue', alpha=0.7, label='d_ff = 4×d_model')
ax_twin.plot(range(len(d_models)), [p/1e6 for p in params_ffn], 'ro-', ms=8, lw=2, label='FFN parametresi (M)')
axes[2].set_xticks(range(len(d_models)))
axes[2].set_xticklabels([str(d) for d in d_models], rotation=30)
axes[2].set_xlabel('d_model'); axes[2].set_ylabel('d_ff', color='steelblue')
ax_twin.set_ylabel('FFN Parametreleri (M)', color='red')
axes[2].set_title('FFN 4x Genişleme Kuralı\nParametreler hızla büyüyor!')
axes[2].legend(loc='upper left'); ax_twin.legend(loc='lower right')

plt.tight_layout()
plt.show()

# FFN implementasyonu
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, activation='gelu', dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.W1      = nn.Linear(d_model, d_ff)
        self.W2      = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.act_fn  = {'relu': F.relu, 'gelu': F.gelu, 'silu': F.silu}[activation]

    def forward(self, x):
        return self.W2(self.dropout(self.act_fn(self.W1(x))))

ffn_demo = FeedForward(d_model=64, activation='gelu')
x_in     = torch.randn(2, 10, 64)
x_out    = ffn_demo(x_in)
print(f'FFN: {tuple(x_in.shape)} → {tuple(x_out.shape)}')
print(f'FFN parametreleri: {sum(p.numel() for p in ffn_demo.parameters()):,}')

---
## Bölüm 8: Layer Normalization — LayerNorm vs RMSNorm, Pre-LN vs Post-LN

$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_j^2 + \epsilon}$$

In [ ]:
# -----------------------------------------------------------
# 8.1 — LayerNorm ve RMSNorm Implementasyonu
# -----------------------------------------------------------
class LayerNorm(nn.Module):
    """LN(x) = γ * (x - μ) / sqrt(σ² + ε) + β"""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.beta  = nn.Parameter(torch.zeros(d))
        self.eps   = eps

    def forward(self, x):
        mu    = x.mean(dim=-1, keepdim=True)
        sigma = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(sigma + self.eps)
        return self.gamma * x_hat + self.beta

class RMSNorm(nn.Module):
    """
    RMSNorm(x) = x / RMS(x) * γ
    β yok! → daha hızlı (~10-15%)
    LLaMA-2/3, Mistral, Falcon'da kullanılıyor
    """
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.eps   = eps

    def forward(self, x):
        rms   = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * x / rms

# -----------------------------------------------------------
# 8.2 — Sayısal Karşılaştırma
# -----------------------------------------------------------
d = 64
ln  = LayerNorm(d)
rms = RMSNorm(d)

# Farklı ölçekte girdilerle test
scales   = [0.01, 0.1, 1.0, 10.0, 100.0]
ln_stds  = []; rms_stds = []; ln_means = []; rms_means = []

for s in scales:
    x_test = torch.randn(32, 10, d) * s
    y_ln   = ln(x_test);  y_rms = rms(x_test)
    ln_stds.append(y_ln.std().item());   rms_stds.append(y_rms.std().item())
    ln_means.append(y_ln.mean().item()); rms_means.append(y_rms.mean().item())

print('Normalizasyon Karşılaştırması:')
print(f'{"Girdi Ölçeği":>15} {"LN std":>10} {"RMS std":>10} {"LN mean":>10} {"RMS mean":>10}')
for s, ls, rs, lm, rm in zip(scales, ln_stds, rms_stds, ln_means, rms_means):
    print(f'{s:>15.2f} {ls:>10.4f} {rs:>10.4f} {lm:>10.4f} {rm:>10.4f}')

# -----------------------------------------------------------
# 8.3 — Pre-LN vs Post-LN: Gradyan Akışı
# -----------------------------------------------------------
class TransformerBlockPostLN(nn.Module):
    """Post-LN (Orijinal): x' = LN(x + SubLayer(x))"""
    def __init__(self, d, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d, n_heads, dropout=0.0)
        self.ffn  = FeedForward(d, dropout=0.0)
        self.ln1  = LayerNorm(d)
        self.ln2  = LayerNorm(d)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.ln1(x + attn_out)   # Post: önce topla, sonra normalize
        x = self.ln2(x + self.ffn(x))
        return x

class TransformerBlockPreLN(nn.Module):
    """Pre-LN (Modern): x' = x + SubLayer(LN(x))"""
    def __init__(self, d, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d, n_heads, dropout=0.0)
        self.ffn  = FeedForward(d, dropout=0.0)
        self.ln1  = LayerNorm(d)
        self.ln2  = LayerNorm(d)

    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + attn_out             # Pre: önce normalize, sonra topla
        x = x + self.ffn(self.ln2(x))
        return x

# Gradyan büyüklüklerini karşılaştır
d, n_heads, T = 32, 4, 8
post_grads = []; pre_grads = []

for _ in range(30):
    for Model, grad_list in [(TransformerBlockPostLN, post_grads),
                              (TransformerBlockPreLN,  pre_grads)]:
        blk    = Model(d, n_heads)
        x_in   = torch.randn(2, T, d, requires_grad=True)
        loss   = blk(x_in).mean()
        loss.backward()
        grad_norm = x_in.grad.norm().item()
        grad_list.append(grad_norm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LayerNorm Analizi', fontsize=14, fontweight='bold')

axes[0].hist(post_grads, bins=15, alpha=0.7, color='salmon',    label='Post-LN (Orijinal)')
axes[0].hist(pre_grads,  bins=15, alpha=0.7, color='steelblue', label='Pre-LN (Modern)')
axes[0].set_xlabel('Girdi Gradyan Normu')
axes[0].set_ylabel('Frekans')
axes[0].set_title('Pre-LN vs Post-LN: Gradyan Dağılımı\n'
                  'Pre-LN daha stabil gradyan akışı sağlar!')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].axvline(np.mean(post_grads), color='red',  ls='--', lw=2, label='Post ortalama')
axes[0].axvline(np.mean(pre_grads),  color='blue', ls='--', lw=2, label='Pre ortalama')

# BN vs LN Karşılaştırma
axes[1].axis('off')
comp_data = [
    ['Özellik',          'BatchNorm',           'LayerNorm',       'RMSNorm'],
    ['Normalize edilen', 'Batch boyutu',        'Token özellik',   'Token özellik'],
    ['Dizi uzunluğu',    'Sabit varsayım',      'Esnek',           'Esnek'],
    ['Batch boyutu',     'Büyük gerekir',       'Küçük stabil',    'Küçük stabil'],
    ['Transformer\'de',  '❌ Zayıf',            '✅ Güçlü',        '✅ Daha hızlı'],
    ['β parametresi',    'Evet',                'Evet',            '❌ Yok'],
    ['Kullanım',         'CNN\'ler',            'BERT, GPT-2',     'LLaMA, Mistral'],
]
tbl = axes[1].table(cellText=comp_data[1:], colLabels=comp_data[0],
                    cellLoc='center', loc='center', bbox=[0, 0.05, 1, 0.9])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for j in range(4): tbl[0,j].set_facecolor('#2C3E50'); tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,7):
    for j in range(4): tbl[i,j].set_facecolor('#EBF5FB' if i%2==0 else '#FDFEFE')
axes[1].set_title('Normalizasyon Yöntemleri Karşılaştırması', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Post-LN ort. gradyan: {np.mean(post_grads):.4f} ± {np.std(post_grads):.4f}')
print(f'Pre-LN  ort. gradyan: {np.mean(pre_grads):.4f} ± {np.std(pre_grads):.4f}')
print('→ Pre-LN daha stabil (küçük standart sapma) ✅')

---
## Bölüm 9: Dikkat Maskeleme — Full vs Causal

**Full (Bidirectional):** BERT, RoBERTa — tüm pozisyonlara bakabilir  
**Causal (Sol-Sağ):** GPT serisi — sadece geçmiş görülür

$$M_{ij} = \begin{cases} 0 & i \geq j \\ -\infty & i < j \end{cases} \quad \Rightarrow \quad e'_{ij} = e_{ij} + M_{ij}$$

In [ ]:
# -----------------------------------------------------------
# 9.1 — Maske Türleri
# -----------------------------------------------------------
def make_full_mask(T):
    """Tam (Bidirectional): Her token her tokena bakabilir."""
    return torch.ones(T, T)

def make_causal_mask(T):
    """
    Kausal (Sol-Sağ): Sadece geçmiş görülür.
    M_{ij} = 0 ise i < j (gelecek → -inf → softmax'ta sıfır)
    """
    return torch.tril(torch.ones(T, T))  # alt üçgen

def apply_mask_and_visualize(Q, K, V, mask):
    """Maske uygulanmış attention ağırlıklarını hesapla."""
    d_k    = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)              # (T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    attn = torch.nan_to_num(attn, nan=0.0)         # -inf → nan → 0
    return attn

T = 6
sentence = ['The', 'cat', 'sat', 'on', 'the', 'mat']

Q = torch.randn(T, 16)
K = torch.randn(T, 16)
V = torch.randn(T, 16)

full_mask   = make_full_mask(T)
causal_mask = make_causal_mask(T)

attn_full   = apply_mask_and_visualize(Q, K, V, full_mask)
attn_causal = apply_mask_and_visualize(Q, K, V, causal_mask)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dikkat Maskeleme: Full vs Causal', fontsize=14, fontweight='bold')

# Full attention
im1 = axes[0].imshow(attn_full.detach().numpy(), cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[0].set_title('Full (Bidirectional) Attention\nBERT, RoBERTa — Tüm pozisyonlar görülür')
axes[0].set_xticks(range(T)); axes[0].set_xticklabels(sentence, fontsize=10)
axes[0].set_yticks(range(T)); axes[0].set_yticklabels(sentence, fontsize=10)
axes[0].set_xlabel('Key (K)'); axes[0].set_ylabel('Query (Q)')
for i in range(T):
    for j in range(T):
        v = attn_full[i,j].item()
        axes[0].text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                     color='white' if v > 0.5 else 'black')
plt.colorbar(im1, ax=axes[0])

# Causal attention
im2 = axes[1].imshow(attn_causal.detach().numpy(), cmap='Reds', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Causal (Sol-Sağ) Attention\nGPT serisi — Sadece geçmiş görülür')
axes[1].set_xticks(range(T)); axes[1].set_xticklabels(sentence, fontsize=10)
axes[1].set_yticks(range(T)); axes[1].set_yticklabels(sentence, fontsize=10)
axes[1].set_xlabel('Key (K)'); axes[1].set_ylabel('Query (Q)')
for i in range(T):
    for j in range(T):
        v = attn_causal[i,j].item()
        if v > 0.001:
            axes[1].text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                         color='white' if v > 0.5 else 'black')
        else:
            axes[1].text(j, i, '−∞', ha='center', va='center', fontsize=8, color='gray')
plt.colorbar(im2, ax=axes[1])

# Uygulama tablosu
axes[2].axis('off')
app_data = [
    ['Maskeleme',  'Model Ailesi',     'Görev',              'Örnek'],
    ['Tam (Bidi)','Encoder (BERT)',    'Anlama, Sınıf.',     'Soru-cevap'],
    ['Causal',    'Decoder (GPT)',     'Metin üretimi',      'ChatGPT'],
    ['Karma',     'Enc-Dec (T5,BART)', 'Seq2Seq, Özetleme',  'Makine çevirisi'],
]
tbl2 = axes[2].table(cellText=app_data[1:], colLabels=app_data[0],
                     cellLoc='center', loc='center', bbox=[0, 0.4, 1, 0.5])
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11)
for j in range(4): tbl2[0,j].set_facecolor('#2C3E50'); tbl2[0,j].set_text_props(color='white', fontweight='bold')
colors_tbl = [('#D6EAF8','#D6EAF8'), ('#FADBD8','#FADBD8'), ('#D5F5E3','#D5F5E3')]
for i in range(1,4):
    for j in range(4): tbl2[i,j].set_facecolor(colors_tbl[i-1][0])

axes[2].text(0.5, 0.25, '💡 Kausal maskede -inf → softmax\'ta sıfır ağırlık',
             ha='center', fontsize=11, color='darkred', transform=axes[2].transAxes,
             bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))
axes[2].text(0.5, 0.10, 'Decoder: "Henüz üretilmemiş kelimeler görülmez"',
             ha='center', fontsize=11, color='darkblue', transform=axes[2].transAxes,
             style='italic')
axes[2].set_title('Maskeleme → Model Seçimi → Görev', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Bölüm 10: Tam Transformer Encoder Bloğu

**Her Encoder Bloğu:**
1. Multi-Head Self-Attention
2. Add & Norm (Residual + LayerNorm)
3. Feed-Forward Network
4. Add & Norm

$$x' = \text{LayerNorm}(x + \text{SubLayer}(x))$$

In [ ]:
# -----------------------------------------------------------
# 10.1 — Tam Transformer Encoder Bloğu
# -----------------------------------------------------------
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN Encoder Bloğu (Modern yaklaşım):
    x' = x + MHA(LN(x))
    x' = x' + FFN(LN(x'))
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.ln1  = LayerNorm(d_model)
        self.ln2  = LayerNorm(d_model)
        self.mha  = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn  = FeedForward(d_model, d_ff, activation='gelu', dropout=dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Residual + MHA
        attn_out, attn_w = self.mha(self.ln1(x), self.ln1(x), self.ln1(x), mask)
        x = x + self.drop(attn_out)

        # Residual + FFN
        x = x + self.drop(self.ffn(self.ln2(x)))

        return x, attn_w

class TransformerEncoder(nn.Module):
    """N katman Encoder."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model    = d_model
        self.embedding  = nn.Embedding(vocab_size, d_model)
        self.pos_enc    = nn.Parameter(torch.zeros(1, max_len, d_model))  # Öğrenilen PE
        self.dropout    = nn.Dropout(dropout)
        self.layers     = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.ln_final   = LayerNorm(d_model)

    def forward(self, x, mask=None):
        B, T = x.shape
        tok_emb = self.embedding(x) * math.sqrt(self.d_model)
        pos_emb = self.pos_enc[:, :T, :]
        h = self.dropout(tok_emb + pos_emb)

        all_attn = []
        for layer in self.layers:
            h, attn_w = layer(h, mask)
            all_attn.append(attn_w)

        return self.ln_final(h), all_attn

# -----------------------------------------------------------
# 10.2 — Test & Parametre Analizi
# -----------------------------------------------------------
configs = [
    {'name': 'Küçük',         'd': 64,   'h': 4,  'L': 2},
    {'name': 'BERT-mini',     'd': 256,  'h': 4,  'L': 4},
    {'name': 'BERT-base (~)', 'd': 256,  'h': 8,  'L': 8},
]

print('Transformer Encoder Parametre Analizi')
print('='*60)
print(f'{"Model":<20} {"d_model":<10} {"Başlar":<10} {"Katman":<10} {"Parametre":<12}')
print('-'*60)

for cfg in configs:
    model  = TransformerEncoder(vocab_size=1000, d_model=cfg['d'],
                                 n_heads=cfg['h'], n_layers=cfg['L'], max_len=128)
    params = sum(p.numel() for p in model.parameters())
    print(f'{cfg["name"]:<20} {cfg["d"]:<10} {cfg["h"]:<10} {cfg["L"]:<10} {params:>12,}')

    # Forward test
    x_ids  = torch.randint(0, 1000, (2, 32))
    out, _ = model(x_ids)

print(f'\nÇıktı boyutu: {tuple(out.shape)} (batch=2, T=32, d={cfg["d"]})')

# Parametre tahmini formülü
print('\nParametre Sayısı Tahmini Formülü:')
print('#params ≈ 12 × N × d²_m')
for N, dm in [(12, 768), (24, 1024), (96, 12288)]:
    est = 12 * N * dm**2
    print(f'  N={N:3d}, d_m={dm:6d} → ~{est/1e6:.0f}M parametre')

---
## Bölüm 11: Mini GPT — Karakter Seviyesi Dil Modeli

GPT = **Decoder-Only Transformer** + Causal Masking + Sonraki token tahmini

Tiny Shakespeare verisi ile küçük bir GPT eğitip metin üretelim!

In [ ]:
# -----------------------------------------------------------
# 11.1 — Mini GPT Decoder Bloğu
# -----------------------------------------------------------
class GPTDecoderBlock(nn.Module):
    """GPT Decoder Bloğu: Causal MHA + FFN + Pre-LN"""
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.mha  = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn  = FeedForward(d_model, d_ff or 4*d_model, activation='gelu', dropout=dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        T = x.shape[1]
        # Causal maske: alt üçgen
        causal = torch.tril(torch.ones(T, T, device=x.device))

        xn = self.ln1(x)
        attn_out, _ = self.mha(xn, xn, xn, causal)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    """Küçük GPT: Karakter seviyesi dil modeli."""
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=3,
                 ctx_len=64, dropout=0.1):
        super().__init__()
        self.ctx_len    = ctx_len
        self.tok_emb    = nn.Embedding(vocab_size, d_model)
        self.pos_emb    = nn.Embedding(ctx_len, d_model)
        self.drop       = nn.Dropout(dropout)
        self.blocks     = nn.Sequential(*[
            GPTDecoderBlock(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.ln_f       = nn.LayerNorm(d_model)
        self.lm_head    = nn.Linear(d_model, vocab_size, bias=False)

        # Ağırlık bağlama: embedding ve lm_head aynı ağırlık
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)

        x  = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x  = self.blocks(x)
        x  = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new=100, temperature=1.0, top_k=None):
        """Otoregresif metin üretimi."""
        self.eval()
        for _ in range(max_new):
            idx_cond  = idx[:, -self.ctx_len:]          # son ctx_len token
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature  # son token logitleri

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float('-inf')

            probs  = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx    = torch.cat([idx, idx_next], dim=1)

        return idx

# -----------------------------------------------------------
# 11.2 — Veri Hazırlama: Küçük Türkçe metin
# -----------------------------------------------------------
text = """
Transformer mimarisi, 2017 yılında Google tarafından tanıtıldı.
Dikkat mekanizması, her token'ın diğer tüm tokenlarla ilişkisini öğrenir.
Büyük dil modelleri bu temel üzerine inşa edilmiştir.
BERT encoder, GPT decoder mimarisi kullanır.
Multi-head attention farklı örüntüleri paralel öğrenir.
Pozisyonel kodlama sıra bilgisini modele ekler.
Feed-forward ağ her token için bağımsız çalışır.
Layer normalization eğitimi stabilize eder.
Residual bağlantılar gradyan akışını kolaylaştırır.
Transformer dil modellemede devrim yarattı.
Yapay zeka alanında büyük ilerlemeler kaydedildi.
Dil modelleri metin üretebilir, çevirebilir, özetleyebilir.
Otoregresif modeller kelime kelime metin üretir.
Dikkat ağırlıkları yorumlanabilir temsiller sağlar.
Modern modeller milyarlarca parametre içerir.
""".strip() * 8  # Küçük eğitim için tekrarla

chars    = sorted(list(set(text)))
vocab_sz = len(chars)
stoi     = {c:i for i,c in enumerate(chars)}
itos     = {i:c for i,c in enumerate(chars)}
encode   = lambda s: [stoi[c] for c in s if c in stoi]
decode   = lambda l: ''.join([itos[i] for i in l])

data     = torch.tensor(encode(text), dtype=torch.long)
split    = int(0.9 * len(data))
train_d  = data[:split]
val_d    = data[split:]

print(f'Karakter sayısı: {vocab_sz}')
print(f'Eğitim token: {len(train_d)}, Doğrulama token: {len(val_d)}')
print(f'Örnek karakterler: {chars[:20]}')

# -----------------------------------------------------------
# 11.3 — Mini GPT Eğitimi
# -----------------------------------------------------------
CTX_LEN  = 48
BATCH_SZ = 32
EPOCHS   = 500

def get_batch(split_data, ctx_len=CTX_LEN, batch_size=BATCH_SZ):
    ix  = torch.randint(len(split_data) - ctx_len, (batch_size,))
    x   = torch.stack([split_data[i:i+ctx_len]   for i in ix])
    y   = torch.stack([split_data[i+1:i+ctx_len+1] for i in ix])
    return x.to(device), y.to(device)

gpt = MiniGPT(vocab_size=vocab_sz, d_model=64, n_heads=4,
              n_layers=3, ctx_len=CTX_LEN, dropout=0.1).to(device)

opt = optim.AdamW(gpt.parameters(), lr=3e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

train_losses = []; val_losses = []

for step in range(EPOCHS):
    gpt.train()
    xb, yb     = get_batch(train_d)
    _, loss    = gpt(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step(); scheduler.step()

    if step % 50 == 0 or step == EPOCHS-1:
        gpt.eval()
        with torch.no_grad():
            xv, yv = get_batch(val_d)
            _, vl  = gpt(xv, yv)
        train_losses.append(loss.item())
        val_losses.append(vl.item())
        print(f'Adım {step:4d} | Eğitim: {loss.item():.4f} | Doğrulama: {vl.item():.4f}')

print('\nMini GPT eğitimi tamamlandı! ✅')

In [ ]:
# -----------------------------------------------------------
# 11.4 — Metin Üretimi & Görselleştirme
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Mini GPT Sonuçları', fontsize=14, fontweight='bold')

# Kayıp eğrileri
steps_plot = list(range(0, EPOCHS, 50))
if (EPOCHS - 1) not in steps_plot:
    steps_plot.append(EPOCHS - 1)
axes[0].plot(steps_plot[:len(train_losses)], train_losses, 'b-o', ms=5, lw=2, label='Eğitim Kaybı')
axes[0].plot(steps_plot[:len(val_losses)],   val_losses,   'r-s', ms=5, lw=2, label='Doğrulama Kaybı')
axes[0].set_xlabel('Adım'); axes[0].set_ylabel('Cross-Entropy Kaybı')
axes[0].set_title('Eğitim Eğrisi\nAzalma = Model öğreniyor!')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Son katmanın dikkat haritası
gpt.eval()
sample_text = 'Transformer'
enc         = torch.tensor([encode(sample_text)], dtype=torch.long).to(device)
if enc.shape[1] > 0:
    with torch.no_grad():
        # Son bloğun dikkat ağırlıklarını al
        xn = gpt.tok_emb(enc) + gpt.pos_emb(torch.arange(enc.shape[1], device=device))
        for i, blk in enumerate(gpt.blocks):
            T_s = xn.shape[1]
            causal = torch.tril(torch.ones(T_s, T_s, device=device))
            xln = blk.ln1(xn)
            attn_out, attn_w = blk.mha(xln, xln, xln, causal)
            # Çift forward kaçınmak için residual adımlarını elle uygula
            xn = xn + blk.drop(attn_out)
            xn = xn + blk.drop(blk.ffn(blk.ln2(xn)))

    attn_show = attn_w[0, 0].cpu().numpy()  # Baş 0
    axes[1].imshow(attn_show, cmap='Blues', aspect='auto')
    labels = list(sample_text)
    axes[1].set_xticks(range(len(labels))); axes[1].set_xticklabels(labels)
    axes[1].set_yticks(range(len(labels))); axes[1].set_yticklabels(labels)
    axes[1].set_title(f'"Transformer" için Dikkat Haritası\n(Son Katman, Baş 0, Causal)')

plt.tight_layout()
plt.show()

# Farklı temperature ile metin üret
print('\n' + '='*60)
print('MİNİ GPT — ÜRETILEN METİNLER')
print('='*60)

for temp, label in [(0.5,'Düşük (0.5) — Daha tutarlı'),
                    (1.0,'Orta   (1.0) — Dengeli'),
                    (1.5,'Yüksek (1.5) — Daha yaratıcı')]:
    seed    = torch.tensor([[stoi.get('T', 0)]], dtype=torch.long).to(device)
    out_ids = gpt.generate(seed, max_new=80, temperature=temp, top_k=10)
    generated = decode(out_ids[0].cpu().tolist())
    print(f'\n📌 Temperature {label}:')
    print(f'   {generated[:120]}')

---
## Bölüm 12: Hiperparametre Analizi & Ölçekleme Yasası

Büyük modellerin tipik değerleri ve **Chinchilla Ölçekleme Yasası**:
> Model büyüklüğü, veri miktarı ve hesaplama bütçesi **birlikte** optimize edilmeli!

In [ ]:
# -----------------------------------------------------------
# 12.1 — Gerçek LLM Hiperparametre Tablosu
# -----------------------------------------------------------
models = {
    'BERT-base':    {'dm': 768,   'N': 12, 'h': 12, 'dff': 3072,  'params_M': 110},
    'BERT-large':   {'dm': 1024,  'N': 24, 'h': 16, 'dff': 4096,  'params_M': 340},
    'GPT-2 (S)':    {'dm': 768,   'N': 12, 'h': 12, 'dff': 3072,  'params_M': 117},
    'GPT-3':        {'dm': 12288, 'N': 96, 'h': 96, 'dff': 49152, 'params_M': 175000},
    'LLaMA-2 7B':   {'dm': 4096,  'N': 32, 'h': 32, 'dff': 11008, 'params_M': 7000},
    'LLaMA-2 70B':  {'dm': 8192,  'N': 80, 'h': 64, 'dff': 28672, 'params_M': 70000},
}

print('Gerçek LLM Hiperparametreleri')
print('='*75)
print(f'{"Model":<15} {"d_m":<8} {"N":<5} {"h":<5} {"d_ff":<8} {"Ger.Par(M)":<12} {"Tah.Par(M)":<12}')
print('-'*75)

for name, cfg in models.items():
    # #params ≈ 12 * N * d_m^2
    est = 12 * cfg['N'] * cfg['dm']**2 / 1e6
    print(f'{name:<15} {cfg["dm"]:<8} {cfg["N"]:<5} {cfg["h"]:<5} '
          f'{cfg["dff"]:<8} {cfg["params_M"]:>12,}   {est:>12.0f}')

print('\nNot: 12·N·d_m² formülü embedding ve FFN\'yi dışlar — bu yüzden tahmin gerçekten küçük.')

# -----------------------------------------------------------
# 12.2 — Ölçekleme Yasası (Chinchilla)
# -----------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('LLM Ölçekleme Analizi', fontsize=14, fontweight='bold')

# Parametre vs kayıp (güç yasası)
param_counts = np.array([110, 340, 117, 175000, 7000, 70000])  # M
approx_loss  = 3.5 / (param_counts ** 0.076)  # Basitleştirilmiş ölçekleme
model_names  = list(models.keys())

axes[0,0].loglog(param_counts, approx_loss, 'b-', lw=2, alpha=0.5)
for i, (n, l) in enumerate(zip(param_counts, approx_loss)):
    axes[0,0].scatter(n, l, s=100, zorder=5)
    axes[0,0].annotate(model_names[i], (n, l),
                        textcoords='offset points', xytext=(5,3), fontsize=8)
axes[0,0].set_xlabel('Model Parametresi (M, log ölçek)')
axes[0,0].set_ylabel('Yaklaşık Kayıp (log ölçek)')
axes[0,0].set_title('Ölçekleme Yasası: Parametre ↑ → Kayıp ↓\n'
                    'Güç Yasası: L ∝ N^{-0.076}')
axes[0,0].grid(True, alpha=0.3, which='both')

# d_model vs konfigürasyon
dm_vals  = [cfg['dm'] for cfg in models.values()]
n_heads  = [cfg['h']  for cfg in models.values()]
n_layers = [cfg['N']  for cfg in models.values()]

axes[0,1].scatter(dm_vals, n_heads, s=100, c='steelblue', zorder=5)
for i, name in enumerate(model_names):
    axes[0,1].annotate(name, (dm_vals[i], n_heads[i]),
                        textcoords='offset points', xytext=(3,3), fontsize=8)
axes[0,1].set_xlabel('d_model'); axes[0,1].set_ylabel('Baş Sayısı (h)')
axes[0,1].set_title('d_model vs Baş Sayısı\nd_k = d_model / h genellikle 64-128')
axes[0,1].grid(True, alpha=0.3)

# GPT vs BERT: Karar çerçevesi
axes[1,0].axis('off')
gpt_bert_data = [
    ['Özellik',         'GPT (Decoder)',          'BERT (Encoder)'],
    ['Mimari',          'Sadece Decoder',          'Sadece Encoder'],
    ['Eğitim görevi',   'Sonraki kelime tahmini',  'Maskeli kelime tahmini'],
    ['Dikkat yönü',     'Sol-Sağ (Causal)',        'İki Yönlü (Bidi.)'],
    ['Bağlam',          'Sadece geçmiş',           'Tüm dizi'],
    ['Güç noktası',     'Üretim, tamamlama',       'Anlama, sınıflandırma'],
    ['Ön-eğitim',       'BookCorpus (7K kitap)',   'BookCorpus + Wikipedia'],
    ['Örnek kullanım',  'ChatGPT, Copilot',        'Arama, NER, Q&A'],
]
tbl = axes[1,0].table(cellText=gpt_bert_data[1:], colLabels=gpt_bert_data[0],
                       cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for j in range(3): tbl[0,j].set_facecolor('#2C3E50'); tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,8):
    tbl[i,0].set_facecolor('#F0F0F0'); tbl[i,0].set_text_props(fontweight='bold')
    tbl[i,1].set_facecolor('#EBF5FB')  # GPT sütunu
    tbl[i,2].set_facecolor('#FDEBD0')  # BERT sütunu
axes[1,0].set_title('GPT vs BERT: Temel Farklar', fontweight='bold', pad=10)

# Ölçekleme özeti
axes[1,1].axis('off')
summary_items = [
    ('🔑 Temel Formül', 'Attention(Q,K,V) = softmax(QK^T/√dk)V', '#3498DB'),
    ('📐 Parametre Tahmini', '#params ≈ 12 × N × d²_model', '#27AE60'),
    ('⚡ Flash Attention', 'Aynı sonuç, ~3-8x daha hızlı', '#E74C3C'),
    ('🔄 RoPE', 'Göreceli PE, uzun bağlam için ideal', '#8E44AD'),
    ('📏 Chinchilla Yasası', 'Model × Veri optimize edilmeli', '#F39C12'),
    ('🏗️ Modern LLM Bloğu', 'RMSNorm + Pre-LN + SwiGLU + RoPE', '#1ABC9C'),
]
for i, (title, content, color) in enumerate(summary_items):
    y = 0.93 - i * 0.155
    axes[1,1].add_patch(mpatches.FancyBboxPatch([0.02, y-0.06], 0.95, 0.13,
        boxstyle='round,pad=0.01', fc=color, ec='white', alpha=0.15))
    axes[1,1].text(0.5, y+0.02, title,   ha='center', fontsize=10, fontweight='bold', color=color)
    axes[1,1].text(0.5, y-0.03, content, ha='center', fontsize=9, fontfamily='monospace', color='black')

axes[1,1].set_xlim(0,1); axes[1,1].set_ylim(0,1)
axes[1,1].set_title('Ders 4 — Kilit Çıkarımlar', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# DERS ÖZETİ
# -----------------------------------------------------------
print('='*65)
print('🎓 Transformer & LLM — 4. Ders Notebook Tamamlandı!')
print('='*65)

summary = [
    ('1. RNN Sorunu',        'Gradyan kaybolması, sıralı hesap, darboğaz'),
    ('2. Bahdanau Dikkat',   'α_{t,i} = softmax(e_{t,i}),  c_t = Σ α h_i'),
    ('3. Scaled Dot-Prod.',  'Attention(Q,K,V) = softmax(QK^T/√dk)V'),
    ('4. Multi-Head Att.',   'h başlık paralel çalışır, farklı ilişkiler'),
    ('5. Pozisyonel Kod.',   'Sin./Öğrenilen/RoPE/ALiBi — sıra bilgisi'),
    ('6. FFN & Aktivasyon',  'ReLU→GELU→SwiGLU, d_ff≈4×d_model'),
    ('7. LayerNorm',         'LN: μ,σ; RMSNorm: yalnızca RMS (~10% hızlı)'),
    ('8. Pre vs Post LN',    'Pre-LN modern modelde daha stabil gradyan'),
    ('9. Maskeleme',         'Full=BERT, Causal=GPT, Karma=T5/BART'),
    ('10. Mini GPT',         'Decoder-only, causal mask, otoregresif üretim'),
    ('11. Ölçekleme',        '#params≈12·N·d²; Chinchilla: model+veri birlikte'),
]

print()
for topic, detail in summary:
    print(f'  ✅ {topic:<22} → {detail}')

print()
print('Sonraki ders: Fine-tuning, RLHF ve İnstruction Tuning!')